# 48 · SLE against healthy children within one lab · control genes for the lab correction

Reads `expression.rds` of GSE65391 (notebook 03b) and `pseudobulk_counts.rds` of GSE135779 (notebook 22).
Writes `sle_vs_healthy_same_lab.csv` and `control_genes_negative.csv` in `data/run_artifacts/CANDLE_SAVI/`.

**Objective.** Find the genes that do not differ between children with SLE and healthy children when both are
measured in the same lab. These are the negative control genes for the lab correction of notebook 49: in the
pooled data of notebook 46, where SLE, the internal diseases and the healthy children come from different labs,
a difference in these genes is taken as unwanted variation (lab, blood tube, processing), not disease.

**The two same-lab studies.**

| study | samples | platform |
|---|---|---|
| GSE65391 | the first visit of 157 children with SLE, and 46 healthy children (one sample each, as in notebook 06c) | Illumina HumanHT-12 array (notebooks 00-03b) |
| GSE135779 | 33 children with SLE and 11 healthy children | single-cell RNA sequencing summed per child (pseudobulk, notebook 22) |

**Method.**
- GSE65391: limma (Ritchie ME et al. *Nucleic Acids Res* 2015;43:e47) on the log2 intensities of the expressed
  genes (notebook 03b), design `~ group`. The two array batches were corrected in notebook 03.
- GSE135779: limma-voom (Law CW et al. *Genome Biol* 2014;15:R29) on the pseudobulk counts, TMM normalization,
  `filterByExpr`, design `~ group + batch` (the sequencing batch of the study).
- Genes are named by Ensembl identifier; GSE65391 genes are symbols, mapped through NCBI gene_info as in
  notebook 46.

**Rules (approved by Anne).**
- **Negative control gene:** tested in both studies, adjusted p >= 0.5 in both, and |log2 fold change| < 0.2 in
  both; and present among the genes of notebook 46.
- **Confirmed SLE gene:** adjusted p < 0.05 in both studies, with the same direction.

## GSE65391: SLE against healthy children, gene by gene

In [1]:
source("../src/paths.R")
suppressMessages({library(limma); library(edgeR)})
e <- readRDS(art("GSE65391", "expression.rds")); m <- e$meta
h <- m[m$disease == "Healthy", ]; h <- h[order(h$subject, h$visit), ]
keep_s <- c(rownames(m)[m$disease == "SLE" & m$first_visit], rownames(h)[!duplicated(h$subject)])
g1 <- factor(ifelse(m[keep_s, "disease"] == "SLE", "SLE", "healthy"), levels = c("healthy", "SLE"))
X1 <- e$E[e$expressed, keep_s]
fit1 <- eBayes(lmFit(X1, model.matrix(~ g1)))
t1 <- topTable(fit1, coef = "g1SLE", number = Inf, sort.by = "none")
c(genes = nrow(X1), SLE = sum(g1 == "SLE"), healthy = sum(g1 == "healthy"), adj_p_below_0.05 = sum(t1$adj.P.Val < 0.05))

genes              SLE          healthy adj_p_below_0.05 
           13259              157               46             5335

**Explanation**
- `e$E` holds the log2 intensities of GSE65391, one row per gene (notebook 03b); `e$expressed` marks the
  expressed genes; `m` is the sample table.
- `keep_s` takes the first visit of each child with SLE and one sample per healthy child (the earliest visit), as
  in notebook 06c.
- `model.matrix(~ g1)` builds the design: an intercept (healthy) and the SLE difference; `lmFit()` and
  `eBayes()` fit it gene by gene (Smyth GK. *Stat Appl Genet Mol Biol* 2004;3:Article3).
- `topTable(..., coef = "g1SLE")` gives each gene's log2 fold change of SLE against healthy and its
  Benjamini-Hochberg adjusted p.

**Result.** 13,259 expressed genes; 157 children with SLE and 46 healthy children; 5,335 genes have adjusted
p < 0.05.

## GSE135779: SLE against healthy children, gene by gene

In [2]:
pb <- readRDS(art("GSE135779", "pseudobulk_counts.rds")); m2 <- pb$meta
g2 <- factor(ifelse(m2$sle == 1, "SLE", "healthy"), levels = c("healthy", "SLE")); b2 <- factor(m2$Batch)
d2 <- model.matrix(~ g2 + b2)
y2 <- DGEList(pb$counts); k2 <- filterByExpr(y2, d2); y2 <- calcNormFactors(y2[k2, , keep.lib.sizes = FALSE])
fit2 <- eBayes(lmFit(voom(y2, d2), d2))
t2 <- topTable(fit2, coef = "g2SLE", number = Inf, sort.by = "none")
rownames(t2) <- pb$genes$ensembl[match(rownames(t2), rownames(pb$counts))]
c(genes = nrow(t2), SLE = sum(g2 == "SLE"), healthy = sum(g2 == "healthy"), adj_p_below_0.05 = sum(t2$adj.P.Val < 0.05))
table(group = g2, batch = b2)

genes              SLE          healthy adj_p_below_0.05 
           15414               33               11             2222

         batch
group     B1 B2 B3 B4 B5 B6
  healthy  0  1  2  2  2  4
  SLE      8  3  6  6  6  4

**Explanation**
- `pb$counts` holds the pseudobulk counts, genes by children (notebook 22); `pb$genes` the Ensembl identifier and
  symbol of each row; `m2$sle` is 1 for SLE and 0 for healthy.
- `model.matrix(~ g2 + b2)` builds the design: the SLE difference and the sequencing batches of the study. Every
  batch but B1 holds both SLE and healthy children, so the batch terms can be estimated.
- `filterByExpr()`, `calcNormFactors()` (TMM), `voom()`, `lmFit()` and `eBayes()` as in notebook 41;
  `topTable(..., coef = "g2SLE")` gives each gene's log2 fold change of SLE against healthy.
- The rows are renamed by Ensembl identifier.

**Result.** 15,414 genes pass the filter; 33 children with SLE and 11 healthy children; 2,222 genes have adjusted
p < 0.05.

## Name the GSE65391 genes by Ensembl identifier

In [3]:
gi <- read.delim(file.path(NCBI, "Homo_sapiens.gene_info.gz"), quote = "", colClasses = "character")
n_ens <- lengths(regmatches(gi$dbXrefs, gregexpr("Ensembl:ENSG", gi$dbXrefs)))
ens <- ifelse(n_ens == 1, sub(".*Ensembl:(ENSG[0-9]+).*", "\\1", gi$dbXrefs), NA)
t1$ensembl <- ens[match(rownames(t1), gi$Symbol)]
t1 <- t1[!is.na(t1$ensembl) & !duplicated(t1$ensembl), ]
c(GSE65391_genes_with_one_Ensembl_id = nrow(t1))

GSE65391_genes_with_one_Ensembl_id 
                             11895

**Explanation**
- As in notebook 46: `ens` is the one Ensembl identifier listed for each NCBI gene; each GSE65391 symbol is
  matched to it. Genes without one, and a second symbol for the same identifier, are dropped.

**Result.** 11,895 GSE65391 genes have one Ensembl identifier.

## Both studies side by side

In [4]:
pool <- readRDS(art("CANDLE_SAVI", "autoimmune_vs_healthy.rds"))
both <- intersect(t1$ensembl, rownames(t2))
r <- data.frame(gene_id = both,
                gene_name = pool$genes$gene_name[match(both, pool$genes$gene_id)],
                logFC_GSE65391 = t1$logFC[match(both, t1$ensembl)], adjP_GSE65391 = t1$adj.P.Val[match(both, t1$ensembl)],
                logFC_GSE135779 = t2[both, "logFC"], adjP_GSE135779 = t2[both, "adj.P.Val"],
                in_pooled_data = both %in% rownames(pool$logcpm))
r$negative_control <- r$adjP_GSE65391 >= 0.5 & r$adjP_GSE135779 >= 0.5 &
                      abs(r$logFC_GSE65391) < 0.2 & abs(r$logFC_GSE135779) < 0.2 & r$in_pooled_data
r$confirmed_SLE <- ifelse(r$adjP_GSE65391 < 0.05 & r$adjP_GSE135779 < 0.05 & sign(r$logFC_GSE65391) == sign(r$logFC_GSE135779),
                          ifelse(r$logFC_GSE65391 > 0, "up", "down"), "")
write.csv(r, art("CANDLE_SAVI", "sle_vs_healthy_same_lab.csv"), row.names = FALSE)
write.csv(r[r$negative_control, c("gene_id", "gene_name")], art("CANDLE_SAVI", "control_genes_negative.csv"), row.names = FALSE)
c(genes_in_both_studies = length(both), in_pooled_data = sum(r$in_pooled_data), negative_controls = sum(r$negative_control),
  confirmed_SLE_up = sum(r$confirmed_SLE == "up"), confirmed_SLE_down = sum(r$confirmed_SLE == "down"),
  spearman_logFC_between_studies = round(cor(r$logFC_GSE65391, r$logFC_GSE135779, method = "spearman"), 2))

genes_in_both_studies                 in_pooled_data 
                       9971.00                        9798.00 
             negative_controls               confirmed_SLE_up 
                        777.00                         384.00 
            confirmed_SLE_down spearman_logFC_between_studies 
                        415.00                           0.42

**Explanation**
- `pool` is the pooled data of notebook 46; its genes decide which controls can be used there.
- `both` holds the genes tested in both studies; `r` puts both studies' log2 fold change and adjusted p side by
  side, with the gene name.
- `negative_control` applies the approved rule; `confirmed_SLE` marks genes with adjusted p < 0.05 in both
  studies and the same direction.
- The full table and the negative control genes are written as CSV files.
- The last line counts genes, controls and confirmed SLE genes, and gives the Spearman correlation of the two
  studies' log2 fold changes over all shared genes.

**Result.** 9,971 genes are tested in both studies; 9,798 of them are in the pooled data of notebook 46. **777
genes are negative controls.** 799 genes are confirmed SLE genes: 384 up and 415 down in both studies. Over all
shared genes, the two studies' log2 fold changes correlate at Spearman rho = 0.42.

## The interferon genes in the two studies

In [5]:
irg <- read_gene_set("ifn-response-genes-28.txt"); nfk <- read_gene_set("nfkb-only-control-11-dejesus2020.txt")
iv <- r[r$gene_name %in% c(irg, nfk), c("gene_name", "logFC_GSE65391", "adjP_GSE65391", "logFC_GSE135779", "adjP_GSE135779", "negative_control", "confirmed_SLE")]
iv$set <- ifelse(iv$gene_name %in% irg, "IRG-28", "NF-kB-11"); iv <- iv[order(iv$set, -iv$logFC_GSE65391), ]
transform(iv, logFC_GSE65391 = round(logFC_GSE65391, 2), adjP_GSE65391 = signif(adjP_GSE65391, 2),
          logFC_GSE135779 = round(logFC_GSE135779, 2), adjP_GSE135779 = signif(adjP_GSE135779, 2))

,gene_name,logFC_GSE65391,adjP_GSE65391,logFC_GSE135779,adjP_GSE135779,negative_control,confirmed_SLE,set
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<chr>,<chr>
3812,IFI27,5.18,1.8e-30,6.43,0.0250,FALSE,up,IRG-28
591,IFI44L,3.71,6.1e-30,2.63,0.0170,FALSE,up,IRG-28
9334,RSAD2,3.37,6.8e-28,2.44,0.0280,FALSE,up,IRG-28
3818,IFIT1,2.82,4.7e-27,2.16,0.0460,FALSE,up,IRG-28
9704,ISG15,2.76,8.0e-22,1.74,0.0069,FALSE,up,IRG-28
3820,IFIT3,2.67,3.1e-28,2.23,0.0270,FALSE,up,IRG-28
385,IFI44,2.62,3.1e-28,2.04,0.0048,FALSE,up,IRG-28
2693,IFI6,2.45,8.9e-26,2.12,0.0048,FALSE,up,IRG-28
8949,OASL,2.44,1.7e-25,1.88,0.0110,FALSE,up,IRG-28


**Explanation**
- The table shows, for the 28 interferon response genes and the 11 NF-kB-only controls found in both studies,
  each study's log2 fold change and adjusted p, and whether the gene is a negative control or a confirmed SLE
  gene. These are the positive controls of notebook 49: they should stay up after the correction.

**Result.** 24 of the 28 interferon response genes are measured in both studies (HERC5, HERC6, IFIT5 and OAS3 have
no Perfect or Good probe on the array, notebook 02). 23 of the 24 are confirmed SLE genes, up in both studies;
SOCS1 is up only in GSE65391. None is a negative control. Of the 8 NF-kB-only controls measured in both, TLR2 and
TANK are up in both studies; none is a negative control.

## References

1. Ritchie ME, Phipson B, Wu D, Hu Y, Law CW, Shi W, Smyth GK. limma powers differential expression analyses for RNA-sequencing and microarray studies. *Nucleic Acids Res* 2015;43:e47.
2. Smyth GK. Linear models and empirical Bayes methods for assessing differential expression in microarray experiments. *Stat Appl Genet Mol Biol* 2004;3:Article3.
3. Law CW, Chen Y, Shi W, Smyth GK. voom: precision weights unlock linear model analysis tools for RNA-seq read counts. *Genome Biol* 2014;15:R29.
4. Robinson MD, McCarthy DJ, Smyth GK. edgeR: a Bioconductor package for differential expression analysis of digital gene expression data. *Bioinformatics* 2010;26:139-140.
5. Robinson MD, Oshlack A. A scaling normalization method for differential expression analysis of RNA-seq data. *Genome Biol* 2010;11:R25.
6. Chen Y, Lun ATL, Smyth GK. From reads to genes to pathways: differential expression analysis of RNA-Seq experiments using Rsubread and the edgeR quasi-likelihood pipeline. *F1000Research* 2016;5:1438.
7. Benjamini Y, Hochberg Y. Controlling the false discovery rate: a practical and powerful approach to multiple testing. *J R Stat Soc B* 1995;57:289-300.
8. Brown GR, Hem V, Katz KS, et al. Gene: a gene-centered information resource at NCBI. *Nucleic Acids Res* 2015;43:D36-D42.
9. Banchereau R, Hong S, Cantarel B, et al. Personalized immunomonitoring uncovers molecular networks that stratify lupus patients. *Cell* 2016;165:551-565.
10. Nehar-Belaid D, Hong S, Marches R, et al. Mapping systemic lupus erythematosus heterogeneity at the single-cell level. *Nat Immunol* 2020;21:1094-1106.
11. Gagnon-Bartsch JA, Speed TP. Using control genes to correct for unwanted variation in microarray data. *Biostatistics* 2012;13:539-552.
12. Risso D, Ngai J, Speed TP, Dudoit S. Normalization of RNA-seq data using factor analysis of control genes or samples. *Nat Biotechnol* 2014;32:896-902.